Revenue Optimization & Risk-Aware Churn Modeling for a Streaming Platform

The original dataset contains user-level churn information in a single flat table.
To better simulate a production streaming environment, this notebook constructs a multi-table relational structure including user activity, transactions, experiments, and support interactions.

In [8]:
import numpy as np
import pandas as pd

In [43]:
users = pd.read_csv("/Users/palakkakani/Desktop/ML/data/users.csv")

In [44]:
users.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
#I rename columns to clearer business friendly names
users = users.rename(columns={
    "customerID": "user_id",
    "tenure": "months_active",
    "Contract": "subscription_type",
    "MonthlyCharges": "monthly_price",
    "TotalCharges": "total_revenue",
    "Churn": "churn_label"
})

In [46]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            7043 non-null   object 
 1   gender             7043 non-null   object 
 2   SeniorCitizen      7043 non-null   int64  
 3   Partner            7043 non-null   object 
 4   Dependents         7043 non-null   object 
 5   months_active      7043 non-null   int64  
 6   PhoneService       7043 non-null   object 
 7   MultipleLines      7043 non-null   object 
 8   InternetService    7043 non-null   object 
 9   OnlineSecurity     7043 non-null   object 
 10  OnlineBackup       7043 non-null   object 
 11  DeviceProtection   7043 non-null   object 
 12  TechSupport        7043 non-null   object 
 13  StreamingTV        7043 non-null   object 
 14  StreamingMovies    7043 non-null   object 
 15  subscription_type  7043 non-null   object 
 16  PaperlessBilling   7043 

In [47]:
users['total_revenue'] = pd.to_numeric(users['total_revenue'], errors='coerce')

In [ ]:
# convert the churn label into a binary format
users['churn_label'] = users['churn_label'].map({'Yes': 1, 'No': 0})

In [49]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            7043 non-null   object 
 1   gender             7043 non-null   object 
 2   SeniorCitizen      7043 non-null   int64  
 3   Partner            7043 non-null   object 
 4   Dependents         7043 non-null   object 
 5   months_active      7043 non-null   int64  
 6   PhoneService       7043 non-null   object 
 7   MultipleLines      7043 non-null   object 
 8   InternetService    7043 non-null   object 
 9   OnlineSecurity     7043 non-null   object 
 10  OnlineBackup       7043 non-null   object 
 11  DeviceProtection   7043 non-null   object 
 12  TechSupport        7043 non-null   object 
 13  StreamingTV        7043 non-null   object 
 14  StreamingMovies    7043 non-null   object 
 15  subscription_type  7043 non-null   object 
 16  PaperlessBilling   7043 

Creating new dataframes

User Activity Table:

In [ ]:
#To make the dataset more meaningful, I introduce realistic behavior patterns.
user_activity = pd.DataFrame()
user_activity['user_id'] = users['user_id']

In [17]:
user_activity['sessions_per_week'] = np.random.poisson(lam=5, size=len(users))

In [18]:
user_activity['avg_watch_minutes'] = np.random.normal(loc=40, scale=15, size=len(users)).clip(5, 200)

In [19]:
user_activity['days_since_last_login'] = np.random.randint(0, 30, size=len(users))

In [20]:
user_activity['preferred_genre'] = np.random.choice(
    ['Action','Comedy','Drama','Horror','Romance'],
    size=len(users)
)

In [21]:
user_activity['device_type'] = np.random.choice(
    ['Mobile','Desktop','SmartTV'],
    size=len(users)
)

In [22]:
user_activity['number_of_profiles'] = np.random.choice([1,2,3,4], size=len(users))

In [ ]:
#Users who churn typically show declining engagement before leaving.
#So I intentionally reduce session counts and watch time for churned users.
user_activity.loc[users['churn_label']==1, 'sessions_per_week'] = \
    user_activity.loc[users['churn_label']==1, 'sessions_per_week'].clip(0,3)
user_activity.loc[users['churn_label']==1, 'avg_watch_minutes'] = \
    user_activity.loc[users['churn_label']==1, 'avg_watch_minutes'].clip(5,30)

In [24]:
user_activity.head()

,user_id,sessions_per_week,avg_watch_minutes,days_since_last_login,preferred_genre,device_type,number_of_profiles
0,7590-VHVEG,3,42.947762,23,Romance,Mobile,2
1,5575-GNVDE,6,50.065239,29,Drama,SmartTV,1
2,3668-QPYBK,2,18.172359,15,Drama,Mobile,4
3,7795-CFOCW,8,52.680470,17,Horror,Desktop,3
4,9237-HQITU,3,34.592972,10,Drama,Desktop,3


Transactions Table

These financial risk signals help model churn in a revenue-aware way

In [25]:
transactions = pd.DataFrame()
transactions['user_id'] = users['user_id']

In [26]:
transactions['billing_cycle'] = np.random.randint(1, 25, size=len(users))

In [27]:
transactions['payment_success'] = np.random.choice([1,0], size=len(users), p=[0.95,0.05])


In [28]:
transactions['failed_payment_count'] = np.random.poisson(lam=0.1, size=len(users))

In [ ]:
transactions['chargeback_flag'] = np.random.choice([0,1], size=len(users), p=[0.98,0.02])

In [30]:
transactions['monthly_revenue'] = users['monthly_price'] * transactions['payment_success']

In [31]:
transactions.head()

,user_id,billing_cycle,payment_success,failed_payment_count,chargeback_flag,monthly_revenue
0,7590-VHVEG,8,1,0,0,29.85
1,5575-GNVDE,9,1,0,0,56.95
2,3668-QPYBK,11,0,0,0,0.00
3,7795-CFOCW,22,1,0,0,42.30
4,9237-HQITU,8,1,0,0,70.70


Experiment Assignment Table : Streaming platforms frequently run promotions and discounts.
This will later allow me to perform experiment analysis and measure business impact.

In [32]:
experiment_assignment = pd.DataFrame()
experiment_assignment['user_id'] = users['user_id']
experiment_assignment['experiment_group'] = np.random.choice(['control','discount'], size=len(users))
experiment_assignment['experiment_start_month'] = np.random.randint(1,13, size=len(users))


In [33]:
experiment_assignment.head()

,user_id,experiment_group,experiment_start_month
0,7590-VHVEG,control,2
1,5575-GNVDE,control,4
2,3668-QPYBK,discount,2
3,7795-CFOCW,control,5
4,9237-HQITU,control,5


Support Tickets : Customer frustration often contributes to churn.

In [34]:
support_tickets=pd.DataFrame()

In [35]:
support_tickets['user_id'] = users['user_id']
support_tickets['ticket_count'] = np.random.poisson(lam=0.3, size=len(users))
support_tickets['avg_response_time'] = np.random.normal(loc=24, scale=10, size=len(users)).clip(1,72)
support_tickets['resolved_flag'] = np.random.choice([0,1], size=len(users), p=[0.1,0.9])


In [36]:
support_tickets.head()

,user_id,ticket_count,avg_response_time,resolved_flag
0,7590-VHVEG,0,36.547521,1
1,5575-GNVDE,0,18.625697,1
2,3668-QPYBK,0,22.353249,1
3,7795-CFOCW,0,33.769264,1
4,9237-HQITU,0,44.230280,1


In [50]:
users.to_csv("/Users/palakkakani/Desktop/ML/data/raw/users.csv", index=False)
user_activity.to_csv("/Users/palakkakani/Desktop/ML/data/raw/user_activity.csv", index=False)
transactions.to_csv("/Users/palakkakani/Desktop/ML/data/raw/transactions.csv", index=False)
experiment_assignment.to_csv("/Users/palakkakani/Desktop/ML/data/raw/experiment_assignment.csv", index=False)
support_tickets.to_csv("/Users/palakkakani/Desktop/ML/data/raw/support_tickets.csv", index=False)